# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row** = one content item (page) aggregated over one month. The raw `fact_daily` table has one row per page per day; I collapse to monthly because the decision moment (which pages to review for CTR opportunity) happens at the end of each month, looking at the full month's performance.

**Tables**: `fact_content_daily_performance` for daily metrics, `dim_content` for static page metadata.

**Time window**: month = 2026-03 (all 31 days of March 2026).

**Label/proxy**: CTR opportunity gap - the residual between a page's observed CTR and the expected CTR for its position tier.

**Deliberately excluded**: Trend labels (trend_direction, trend_pct) — they measure directional change, not CTR underperformance.

In [5]:
# Setup + explore the data
# %pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
D = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
C = f"read_parquet('{REL}/dim_content.parquet')"
CL = f"read_parquet('{REL}/dim_clients.parquet')"

print("=== dim_clients sample ===")
con.sql(f"SELECT * FROM {CL} LIMIT 3").show()

print("\n=== dim_content columns + sample ===")
con.sql(f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM {C})").show()
con.sql(f"SELECT * FROM {C} LIMIT 3").show()

print("\n=== fact_daily columns ===")
con.sql(f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM {D})").show()
print("\n=== fact_daily sample row ===")
con.sql(f"SELECT * FROM {D} LIMIT 1").show()

print("\n=== date range ===")
con.sql(f"SELECT MIN(report_date) as first, MAX(report_date) as last FROM {D}").show()

=== dim_clients sample ===
┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false        

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│   first    │    last    │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-01-27 │ 2026-06-30 │
└────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` | Feature | Knowable at decision moment; search appearance volume |
| `gsc_avg_position` | Feature | Knowable at decision moment; average ranking |
| `ga4_sessions` | Feature | Knowable at decision moment; engagement volume |
| `content_type` | Feature | Static metadata — set at page creation, never changes |
| `main_intent` | Feature | Static metadata — set at page creation |
| `word_count` | Feature | Static metadata — known from page content |
| `gsc_clicks` | **Excluded** | Mechanically determines CTR (= clicks / impressions). Including it would leak the label |
| `gsc_ctr` (computed) | **Label / proxy** | The thing I predict — CTR = clicks/impressions |
| `client_hash_id` | Context | Join key only; never a feature |
| `content_hash_id` | Context | Join key only; never a feature |
| `report_date` | Context | Defines the time window; not a model input |
| `month` | Context | Partition label; not a model input |
| `ga4_data_available` | Context | Filter flag; rows before GA4 start are zero-filled |
| `trend_direction` | Excluded | Measures change over time, not current CTR performance |
| `trend_pct` | Excluded | Same reason — derived from trend_direction |
| `provider_used` | Excluded | Irrelevant to CTR — LLM provider used for content generation |
| `model_used` | Excluded | Irrelevant to CTR — LLM model used |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Use the full daily table (not sample) for March 2026
# First: define the March filter
MARCH = "month = '2026-03'"
D = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# --- QUERY 1: Grain check ---
print("=== QUERY 1: GRAIN CHECK ===")
print("Group by (report_date, client_hash_id, content_hash_id) - expect 0 rows with >1")
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS cnt
    FROM {D}
    WHERE {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING cnt > 1
    LIMIT 5
""").show()

# --- QUERY 2: Row count + date span ---
print("\n=== QUERY 2: ROW COUNT + DATE SPAN ===")
con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {D}
    WHERE {MARCH}
""").show()

# --- QUERY 3: Availability (IS TRUE) ---
print("\n=== QUERY 3: AVAILABILITY ===")
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS with_ga4,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS with_gsc,
           SUM(CASE WHEN ga4_data_available AND gsc_data_available THEN 1 ELSE 0 END) AS with_both
    FROM {D}
    WHERE {MARCH}
""").show()

=== QUERY 1: GRAIN CHECK ===
Group by (report_date, client_hash_id, content_hash_id) - expect 0 rows with >1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │  cnt  │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘


=== QUERY 2: ROW COUNT + DATE SPAN ===
┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘


=== QUERY 3: AVAILABILITY ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────┬──────────┬───────────┐
│ total_rows │ with_ga4 │ with_gsc │ with_both │
│   int64    │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼───────────┤
│    9841378 │   413966 │  3611061 │    364347 │
└────────────┴──────────┴──────────┴───────────┘



## 3b. Five features (with available-when lines)

| # | Feature | Available when? |
|---|---|---|
| 1 | **impressions** | Knowable at decision moment because it sums gsc_impressions already observed over the completed month. |
| 2 | **avg_position** | Knowable at decision moment because it is the average gsc_avg_position over the completed month. |
| 3 | **ga4_sessions** | Knowable at decision moment because it sums ga4_sessions already observed over the month (filtering on ga4_data_available = TRUE). |
| 4 | **content_type** | Knowable at decision moment because it is static metadata set at page creation, never changes. |
| 5 | **word_count** | Knowable at decision moment because it is static content metadata measured at page creation. |

In [7]:
# Build the monthly feature frame for Lane 4
D = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
C = f"read_parquet('{REL}/dim_content.parquet')"
MARCH = "month = '2026-03'"

features = con.sql(f"""
SELECT f.content_hash_id,
       f.client_hash_id,
       SUM(f.gsc_impressions) AS impressions,
       AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position ELSE NULL END) AS avg_position,
       SUM(f.ga4_sessions) AS sessions,
       ANY_VALUE(dc.content_type) AS content_type,
       MAX(dc.word_count) AS word_count
FROM {D} f
LEFT JOIN {C} dc ON f.content_hash_id = dc.content_hash_id
WHERE {MARCH}
  AND f.ga4_data_available = TRUE
  AND f.gsc_impressions > 0
GROUP BY f.content_hash_id, f.client_hash_id
HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Feature frame: {len(features):,} content items')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 32,596 content items


,content_hash_id,client_hash_id,impressions,avg_position,sessions,content_type,word_count
0,content_15770c63daac443b,client_2094c6eb080311d5,1688.0,7.002128,5.0,keyword article,2694
1,content_162b33b49be30cb5,client_2094c6eb080311d5,202.0,4.339242,11.0,keyword article,2364
2,content_167ea9e53c1a8e6e,client_2094c6eb080311d5,1937.0,5.896372,10.0,keyword article,2547
3,content_16b0b94cf7e7ca80,client_2094c6eb080311d5,518.0,6.891539,19.0,keyword article,2964
4,content_16c9e8971bd737c3,client_2094c6eb080311d5,188.0,4.005864,15.0,keyword article,1597


## 3c. The trap - deliberate leakage experiment

I add CTR itself as a feature. Since the label IS CTR, feeding CTR as a feature gives the model the answer. Score jumps. Then I delete the leaky column and keep the honest number.

In [10]:
# --- THE TRAP ---
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# Get CTR label
df_lab = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr
    FROM {D} f
    WHERE {MARCH} AND f.ga4_data_available = TRUE AND f.gsc_impressions > 0
    GROUP BY f.content_hash_id HAVING SUM(f.gsc_impressions) >= 100
""").df()

df_full = features.merge(df_lab, on='content_hash_id', how='inner')
df_full = df_full.dropna(subset=['impressions', 'avg_position', 'sessions', 'word_count', 'ctr'])

# Encode categoricals
X_clean = pd.concat([
    df_full[['impressions', 'avg_position', 'sessions', 'word_count']],
    pd.get_dummies(df_full['content_type'], prefix='type')
], axis=1)
y = df_full['ctr'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y, test_size=0.3, random_state=42)

model_clean = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_clean.fit(X_tr, y_tr)
y_pred = model_clean.predict(X_te)
print('=== HONEST MODEL (no leakage) ===')
print(f'R2 = {r2_score(y_te, y_pred):.4f}')

# Now add the leaky column: clicks
df_clicks = con.sql(f"""
    SELECT f.content_hash_id, SUM(f.gsc_clicks) AS clicks
    FROM {D} f
    WHERE {MARCH} AND f.ga4_data_available = TRUE AND f.gsc_impressions > 0
    GROUP BY f.content_hash_id HAVING SUM(f.gsc_impressions) >= 100
""").df()

df_leaky = df_full.merge(df_clicks, on='content_hash_id', how='inner')
X_leaky = pd.concat([
    df_leaky[['impressions', 'avg_position', 'sessions', 'word_count', 'clicks']],
    pd.get_dummies(df_leaky['content_type'], prefix='type')
], axis=1)

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leaky, df_leaky['ctr'].values, test_size=0.3, random_state=42)
model_leaky = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_leaky.fit(Xl_tr, yl_tr)
r2_leaky = r2_score(yl_te, model_leaky.predict(Xl_te))
print('\n=== LEAKY MODEL (clicks added as feature) ===')
print(f'R2 = {r2_leaky:.4f} (inflated!)')
print('\nClicks leaked CTR because CTR = clicks/impressions.')
print('Dropping clicks. Honest R2 restored.')

=== HONEST MODEL (no leakage) ===
R2 = 0.3691

=== LEAKY MODEL (clicks added as feature) ===
R2 = 0.9897 (inflated!)

Clicks leaked CTR because CTR = clicks/impressions.
Dropping clicks. Honest R2 restored.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**1. GA4 data is sparse.** Only 4.2% of March 2026 rows have `ga4_data_available = TRUE`. For Lane 4, which uses sessions as a feature, this means most rows can't contribute engagement metrics — they have zero-filled GA4 columns.

**2. Unbalanced panel.** Per-client history varies. Some clients have 17 months of data, others only a few months. A single-month snapshot ignores this — results may not generalize across clients.

**3. Position = 0 means no data, not rank zero.** Rows where `gsc_avg_position = 0` have no position measurement. These can't be compared in position-tier analysis.

**4. Observational, not causal.** A low CTR ranking means "this page deserves a look" — it does NOT prove that editing the page will raise CTR. That requires an experiment.

**5. Single month only.** March 2026 is one calendar period. Seasonal patterns or events may not represent typical conditions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.